In [2]:
from pathlib import Path
import csv

# --- settings ---
BASE_DIRS = [
     Path("ksa_txt"),
#    Path("dnb_txt"),
#    Path("ksa_txt")
#Path("../../Beschikkingen/DrentheSubsidies_sentences.csv")
]
OUT_CSV = Path("txt_csv/ksa_txt.csv")

def read_text_with_fallback(p: Path) -> str:
    """Try multiple encodings, fall back to replacement if needed."""
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            with p.open("r", encoding=enc, errors="strict") as f:
                return f.read()
        except Exception:
            continue
    with p.open("r", encoding="utf-8", errors="replace") as f:
        return f.read()

def normalize_newlines(s: str) -> str:
    return s.replace("\r\n", "\n").replace("\r", "\n")

# --- collect files ---
rows = []
seen_ids = {}

for base_dir in BASE_DIRS:
    txt_files = sorted(base_dir.rglob("*.txt"))
    if not txt_files:
        print(f"⚠️ No .txt files found under {base_dir.resolve()}")
        continue

    for p in txt_files:
        text = normalize_newlines(read_text_with_fallback(p))
        base_id = p.stem

        # Ensure unique IDs across ALL dirs
        if base_id in seen_ids:
            seen_ids[base_id] += 1
            file_id = f"{base_id}_{seen_ids[base_id]}"
        else:
            seen_ids[base_id] = 0
            file_id = base_id

        rows.append({"id": file_id, "text": text})

# --- write CSV ---
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "text"])
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ Done. Wrote {len(rows)} records to {OUT_CSV.resolve()}")

✅ Done. Wrote 150 records to C:\Users\hnan\OneDrive - Tilburg University\Documents\GitHub\Harry-Nan\txt_csv\ksa_txt.csv


Je bent een juridisch taalmodel gespecialiseerd in het herkennen van besluiten/rechtsgevolgen in beschikkingen (subsidies, vergunningen, ontheffingen en concessies). Elke invoer is één zin. Label elke zin met:
	• True = de zin formuleert het bindende, definitieve rechtsgevolg (zoals het verlenen/afwijzen/wijzigen/intrekken/verhogen/verlagen van subsidie, vergunning, etc., of het toekennen van voorschotten).
	• False = geen bindend rechtsgevolg (zoals motivatie, voorwaarden, berekeningsregels, algemene normen, hypotheses, plichten, procedurele of contextuele mededelingen). 

Output: geef het resultaat als JSON-array in dit formaat:
{ "id": <ID-waarde>, "decision_sentence": true|false }
Verwerk alleen de zinnen die ik je aanlever; elke zin staat op een aparte regel. Controleer je antwoord. Geef een download-link.

In [3]:
import json
from pathlib import Path
from collections import defaultdict, OrderedDict
import string
import glob

# === CONFIG ===
SYSTEM_PATHS = [
    "llm/labels/evaluate/acm_sancties_entities.json",
    "llm/labels/evaluate/acm_txt_entities.json",
    "llm/labels/evaluate/anvs_txt_entities.json",
    "llm/labels/evaluate/dnb_txt_entities.json",
    "llm/labels/evaluate/drenthesubsidies_entities.json",
    "llm/labels/evaluate/erotterdam_entities.json",
    "llm/labels/evaluate/ksa_txt_entities.json",
    "llm/labels/evaluate/rijksoverheid_txt_entities.json"
]

GOLD_GLOB = 'data/CORRECT_*.json'

LABEL_SPECS = OrderedDict({
    'ONTVANGER':   {1, 2},
    'BESLUITVORMEND_ORGAAN':  {3, 4},
    'RECHTSOBJECT':{5, 6},
    'RECHTSHANDELING':   {7, 8},
    'GRONDSLAG': {9, 10},
})

def detokenize(tokens):
    s = ' '.join(tokens)
    s = (s.replace(' ,', ',').replace(' .', '.')
           .replace(' ;', ';').replace(' :', ':')
           .replace(" )", ")").replace("( ", "(")
           .replace(" %", "%").replace("€ ", "€"))
    return s

def extract_gold_entities_from_item_generic(item, label_specs):
    out = {label: set() for label in label_specs}
    tokens = item.get('tokens')
    tags = item.get('ner_tags')
    if not isinstance(tokens, list) or not isinstance(tags, list) or len(tokens) != len(tags):
        return out
    for label, for_tags in label_specs.items():
        current = []
        spans = set()
        for tok, tg in zip(tokens, tags):
            if tg in for_tags:
                current.append(tok)
            else:
                if current:
                    spans.add(' '.join(current))
                    current = []
        if current:
            spans.add(' '.join(current))
        out[label] = spans
    return out

def get_item_id(item, fallback_idx):
    return item.get('id') or item.get('doc_id') or item.get('meta_id') or f'row_{fallback_idx}'

def empty_label_sets():
    return {label: set() for label in LABEL_SPECS.keys()}

def canon_id(x):
    if x is None:
        return None
    s = str(x)
    s = s.rsplit('/', 1)[-1].rsplit('\\', 1)[-1]
    if s.lower().endswith('.txt'):
        s = s[:-4]
    return s

def normalize_span_loose(span: str) -> str:
    """Lowercase, remove punctuation inside, collapse spaces."""
    if not span:
        return span
    s = span.lower()
    for ch in ",;()[]{}":
        s = s.replace(ch, " ")
    s = " ".join(s.split())
    return s.strip(string.punctuation + " ")

def normalize_set(spans):
    return {normalize_span_loose(s) for s in spans if s}

# --- load system predictions from hardcoded list ---
sys_pred_by_id = {}
for sys_path in SYSTEM_PATHS:
    with open(sys_path, 'r', encoding='utf-8') as f:
        system = json.load(f)

    for raw_id, content in system.items():
        cid = canon_id(raw_id)
        if not cid:
            continue
        if cid not in sys_pred_by_id:
            sys_pred_by_id[cid] = empty_label_sets()
        for ent in content.get('entities', []):
            label = ent.get('label')
            text = ent.get('text')
            if not text or not label:
                continue
            label_up = str(label).upper()

            if label_up == "GRONDSLAG":
                lower_text = text.lower()
                if "verleen" in lower_text:
                    cut_idx = lower_text.find("verleen")
                    text = text[:cut_idx].strip()
                    if not text:
                        continue
            if label_up in sys_pred_by_id[cid]:
                sys_pred_by_id[cid][label_up].add(text)

# --- load gold sentences from hardcoded list ---
items = []
for path in glob.glob(GOLD_GLOB):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        items.extend(data)
    elif isinstance(data, dict):
        if 'rows' in data and isinstance(data['rows'], list):
            for r in data['rows']:
                if isinstance(r, dict) and 'tokens' in r and 'ner_tags' in r:
                    items.append(r)
        else:
            for v in data.values():
                if isinstance(v, dict) and 'tokens' in v and 'ner_tags' in v:
                    items.append(v)
                elif isinstance(v, list):
                    for it in v:
                        if isinstance(it, dict) and 'tokens' in it and 'ner_tags' in it:
                            items.append(it)

print(f"Loaded {len(items)} gold sentences")

# --- metrics ---
global_tp = defaultdict(int)
global_fp = defaultdict(int)
global_fn = defaultdict(int)

matched_ids = 0
unmatched_ids = 0

for i, it in enumerate(items, 1):
    tokens = it.get('tokens', [])
    sent_surface = detokenize(tokens)

    sent_id_raw = get_item_id(it, i)
    sent_id_can = canon_id(sent_id_raw)

    gold_by_label = extract_gold_entities_from_item_generic(it, LABEL_SPECS)
    pred_by_label = sys_pred_by_id.get(sent_id_can, empty_label_sets())

    per_label = {}
    any_tp = False
    for label in LABEL_SPECS.keys():
        gold_set_raw = gold_by_label.get(label, set())
        pred_set_raw = pred_by_label.get(label, set())

        if label == "GRONDSLAG":
            gold_set = {normalize_span_loose(s) for s in gold_set_raw if s}
            pred_set = {normalize_span_loose(s) for s in pred_set_raw if s}
            matched_pred = set()
            matched_gold = set()
            for p in pred_set:
                for g in gold_set:
                    if not p or not g:
                        continue
                    if p in g or g in p:
                        matched_pred.add(p)
                        matched_gold.add(g)
                        break
            tp = sorted(matched_pred)
            fp = sorted(pred_set - matched_pred)
            fn = sorted(gold_set - matched_gold)
        else:
            gold_set = normalize_set(gold_set_raw)
            pred_set = normalize_set(pred_set_raw)
            tp = sorted(gold_set & pred_set)
            fp = sorted(pred_set - gold_set)
            fn = sorted(gold_set - pred_set)

        if tp:
            any_tp = True

        per_label[label] = (tp, fp, fn)

    if sent_id_can in sys_pred_by_id:
        matched_ids += 1
    else:
        unmatched_ids += 1

    if not any_tp:
        continue

    print(f"\nsentence: {sent_id_raw}\n {sent_surface}")
    for label in LABEL_SPECS.keys():
        tp, fp, fn = per_label[label]
        print(f"  [{label}]")
        print(f"    TP: {tp}")
        print(f"    FP: {fp}")
        print(f"    FN: {fn}")

        global_tp[label] += len(tp)
        global_fp[label] += len(fp)
        global_fn[label] += len(fn)

def safe_div(n, d): return n / d if d else 0.0

print(f"\nMatched gold IDs in system predictions: {matched_ids}")
print(f"Unmatched gold IDs (no system preds for these): {unmatched_ids}")

print("\n=== Per-label metrics (Precision, Recall, F1) ===")
for label in LABEL_SPECS.keys():
    tp = global_tp[label]; fp = global_fp[label]; fn = global_fn[label]
    P = safe_div(tp, tp + fp); R = safe_div(tp, tp + fn)
    F1 = safe_div(2 * P * R, P + R)
    print(f"{label:12s}  P={P:.3f}  R={R:.3f}  F1={F1:.3f}  (TP={tp}, FP={fp}, FN={fn})")

Loaded 1104 gold sentences

sentence: www.acm.nl_sites_default_files_old_publication_bijlagen_3011_3938_532_533_534BLD.pdf_2.txt
 De Raad legt een boete op van EUR 26.435,00 aan J.H.C. Habets Beheer B.V., gevestigd te Nuth.
  [ONTVANGER]
    TP: []
    FP: ['aan j.h.c. habets beheer b.v']
    FN: ['j.h.c. habets beheer b.v']
  [BESLUITVORMEND_ORGAAN]
    TP: ['de raad']
    FP: []
    FN: []
  [RECHTSOBJECT]
    TP: ['boete']
    FP: []
    FN: []
  [RECHTSHANDELING]
    TP: ['legt']
    FP: []
    FN: []
  [GRONDSLAG]
    TP: []
    FP: []
    FN: []

sentence: www.acm.nl_sites_default_files_old_publication_bijlagen_3024_3938_-_687-704BLD.pdf_2.txt
 De Raad legt een boete op van EUR 46.389,00 aan Holding Maatschappij B.M. van Houwelingen B.V., gevestigd te Hardinxveld Giessendam.
  [ONTVANGER]
    TP: []
    FP: ['aan holding maatschappij b.m. van houwelingen b.v']
    FN: ['holding maatschappij b.m. van houwelingen b.v']
  [BESLUITVORMEND_ORGAAN]
    TP: ['de raad']
    FP: []
    FN

In [5]:
# === Per-authority metrics ===
print("\n=== Per-authority metrics (Precision, Recall, F1) ===")

authority_stats = defaultdict(lambda: defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0}))

# Re-run loop to collect per-authority metrics
for i, it in enumerate(items, 1):
    sent_id_raw = get_item_id(it, i)
    sent_id_can = canon_id(sent_id_raw)
    authority = str(sent_id_raw)[:10]  # first 5 chars

    gold_by_label = extract_gold_entities_from_item_generic(it, LABEL_SPECS)
    pred_by_label = sys_pred_by_id.get(sent_id_can, empty_label_sets())

    for label in LABEL_SPECS.keys():
        gold_set_raw = gold_by_label.get(label, set())
        pred_set_raw = pred_by_label.get(label, set())

        if label == "GRONDSLAG":
            gold_set = {normalize_span_loose(s) for s in gold_set_raw if s}
            pred_set = {normalize_span_loose(s) for s in pred_set_raw if s}
            matched_pred = set()
            matched_gold = set()
            for p in pred_set:
                for g in gold_set:
                    if not p or not g:
                        continue
                    if p in g or g in p:
                        matched_pred.add(p)
                        matched_gold.add(g)
                        break
            tp = len(matched_pred)
            fp = len(pred_set - matched_pred)
            fn = len(gold_set - matched_gold)
        else:
            gold_set = normalize_set(gold_set_raw)
            pred_set = normalize_set(pred_set_raw)
            tp = len(gold_set & pred_set)
            fp = len(pred_set - gold_set)
            fn = len(gold_set - pred_set)

        authority_stats[authority][label]['tp'] += tp
        authority_stats[authority][label]['fp'] += fp
        authority_stats[authority][label]['fn'] += fn

# --- Print per-authority aggregated metrics ---
for auth, stats in sorted(authority_stats.items()):
    print(f"\nAuthority: {auth}")
    for label, counts in stats.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        P = safe_div(tp, tp + fp)
        R = safe_div(tp, tp + fn)
        F1 = safe_div(2 * P * R, P + R)
        print(f"  {label:12s}  P={P:.3f}  R={R:.3f}  F1={F1:.3f}  (TP={tp}, FP={fp}, FN={fn})")



=== Per-authority metrics (Precision, Recall, F1) ===

Authority: kansspelau
  ONTVANGER     P=0.130  R=0.107  F1=0.118  (TP=3, FP=20, FN=25)
  BESLUITVORMEND_ORGAAN  P=0.361  R=0.935  F1=0.521  (TP=43, FP=76, FN=3)
  RECHTSOBJECT  P=0.227  R=0.811  F1=0.355  (TP=30, FP=102, FN=7)
  RECHTSHANDELING  P=0.262  R=0.757  F1=0.389  (TP=28, FP=79, FN=9)
  GRONDSLAG     P=0.800  R=0.480  F1=0.600  (TP=12, FP=3, FN=13)

Authority: open.overh
  ONTVANGER     P=0.770  R=0.940  F1=0.847  (TP=47, FP=14, FN=3)
  BESLUITVORMEND_ORGAAN  P=0.803  R=0.953  F1=0.871  (TP=61, FP=15, FN=3)
  RECHTSOBJECT  P=0.672  R=0.976  F1=0.796  (TP=82, FP=40, FN=2)
  RECHTSHANDELING  P=0.329  R=0.295  F1=0.311  (TP=26, FP=53, FN=62)
  GRONDSLAG     P=1.000  R=0.045  F1=0.087  (TP=1, FP=0, FN=21)

Authority: puc.overhe
  ONTVANGER     P=0.073  R=0.092  F1=0.081  (TP=9, FP=115, FN=89)
  BESLUITVORMEND_ORGAAN  P=0.600  R=1.000  F1=0.750  (TP=3, FP=2, FN=0)
  RECHTSOBJECT  P=0.797  R=0.982  F1=0.880  (TP=110, FP=28, FN=